# v04 — Features derivadas

Partindo do campeão (logística com `elo_diff`), testo adicionar **uma variável por vez**, todas calculadas a partir da própria linha:

- `venue_form_diff` = forma do mandante em casa − forma do visitante fora;
- `form_pts_diff`, `gd_diff` (diferenças de forma e saldo recente);
- variáveis originais isoladas (`form_pts_away`, `h2h_home_pts`, `rest_days_home`...).

Depois combino as que melhoram.

In [1]:
import warnings
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)

DATA = Path('../data')
SUB_DIR = Path('../submissions')
SUB_DIR.mkdir(exist_ok=True)
LOG_PATH = Path('../experimentos.csv')

CLASS_ORDER = ['A', 'D', 'H']
RANDOM_STATE = 42
FOLDS = [2016, 2017, 2018, 2019, 2020, 2021]  # walk-forward: treino = temporadas anteriores

train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
print(train.shape, test.shape)

(3419, 26) (1725, 22)


In [2]:
NUM_FEATURES = ['elo_home', 'elo_away', 'elo_diff', 'form_pts_home', 'form_pts_away',
                'form_gf_home', 'form_ga_home', 'form_gf_away', 'form_ga_away',
                'home_form_as_host', 'away_form_as_visitor', 'rest_days_home', 'rest_days_away',
                'season_ppg_home', 'season_ppg_away', 'round_n', 'h2h_home_pts']


def add_features(df):
    """Features derivadas usando apenas colunas da propria linha."""
    out = df.copy()
    out['venue_form_diff'] = out['home_form_as_host'] - out['away_form_as_visitor']
    out['form_pts_diff'] = out['form_pts_home'] - out['form_pts_away']
    out['gd_diff'] = (out['form_gf_home'] - out['form_ga_home']) - (out['form_gf_away'] - out['form_ga_away'])
    return out


train = add_features(train)
test = add_features(test)

In [3]:
def make_logreg(num_cols, cat_cols=(), C=1.0):
    transformers = [('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                                      ('scale', StandardScaler())]), list(num_cols))]
    if cat_cols:
        transformers.append(('cat', OneHotEncoder(handle_unknown='ignore'), list(cat_cols)))
    return Pipeline([('prep', ColumnTransformer(transformers)),
                     ('model', LogisticRegression(C=C, max_iter=5000, random_state=RANDOM_STATE))])


def predict_ordered(model, X):
    """Alinha predict_proba a ['A', 'D', 'H'] pelas classes do modelo."""
    raw = pd.DataFrame(model.predict_proba(X), columns=model.named_steps['model'].classes_)
    return raw.reindex(columns=CLASS_ORDER, fill_value=0.0).values


def walk_forward(build, cols):
    """Previsoes out-of-fold por temporada; o modelo e reconstruido do zero a cada fold."""
    preds, ys = {}, {}
    for season in FOLDS:
        tr = train[train['Season'] < season]
        va = train[train['Season'] == season]
        model = build()
        model.fit(tr[cols], tr['Res'])
        preds[season] = predict_ordered(model, va[cols])
        ys[season] = va['Res'].values
    return preds, ys


def summarize(preds, ys, name):
    rows = [{'Season': s, 'n': len(ys[s]), 'LogLoss': log_loss(ys[s], preds[s], labels=CLASS_ORDER)}
            for s in FOLDS]
    df = pd.DataFrame(rows)
    mean = np.average(df['LogLoss'], weights=df['n'])
    print(f'{name}: media ponderada = {mean:.4f} | desvio entre folds = {df.LogLoss.std():.4f}')
    return df, mean

In [4]:
def save_submission(proba, path):
    sub = pd.DataFrame(proba, columns=CLASS_ORDER)
    sub.insert(0, 'Id', test['Id'].values)
    assert list(sub.columns) == ['Id', 'A', 'D', 'H']
    assert len(sub) == len(test) and (sub['Id'].values == test['Id'].values).all()
    vals = sub[CLASS_ORDER].values
    assert np.isfinite(vals).all() and (vals > 0).all()
    assert np.allclose(vals.sum(axis=1), 1.0, atol=1e-9)
    sub.to_csv(path, index=False)
    print('salvo em', path, sub.shape)
    return sub


def log_experiment(version, change, features, model_desc, params, folds_df, mean, decision, csv_path):
    row = {'versao': version, 'data': date.today().isoformat(), 'mudanca': change,
           'features': features, 'modelo': model_desc, 'parametros': params,
           'folds': ','.join(map(str, FOLDS)),
           'logloss_por_fold': ';'.join(f'{s}:{v:.4f}' for s, v in zip(folds_df.Season, folds_df.LogLoss)),
           'logloss_medio': round(mean, 4), 'desvio': round(folds_df.LogLoss.std(), 4),
           'decisao': decision, 'csv': csv_path}
    log = pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame(columns=list(row.keys()))
    log = log[log['versao'] != version]
    log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
    log.to_csv(LOG_PATH, index=False)
    return log

In [5]:
base_preds, ys = walk_forward(lambda: make_logreg(['elo_diff'], C=0.1), ['elo_diff'])
base_df, base_mean = summarize(base_preds, ys, 'campeao v03')

rows = []
for extra in ['venue_form_diff', 'form_pts_diff', 'gd_diff', 'form_pts_away', 'form_pts_home',
              'h2h_home_pts', 'rest_days_home', 'season_ppg_home', 'round_n']:
    cols = ['elo_diff', extra]
    preds, _ = walk_forward(lambda: make_logreg(cols, C=0.1), cols)
    df, m = summarize(preds, ys, f'+ {extra}')
    rows.append({'extra': extra, 'media': m,
                 'folds_melhores': int((df.LogLoss.values < base_df.LogLoss.values).sum())})
pd.DataFrame(rows).sort_values('media').round(4)

campeao v03: media ponderada = 1.0228 | desvio entre folds = 0.0414


+ venue_form_diff: media ponderada = 1.0226 | desvio entre folds = 0.0417


+ form_pts_diff: media ponderada = 1.0233 | desvio entre folds = 0.0421


+ gd_diff: media ponderada = 1.0241 | desvio entre folds = 0.0436


+ form_pts_away: media ponderada = 1.0226 | desvio entre folds = 0.0405


+ form_pts_home: media ponderada = 1.0235 | desvio entre folds = 0.0417


+ h2h_home_pts: media ponderada = 1.0232 | desvio entre folds = 0.0417
+ rest_days_home: media ponderada = 1.0241 | desvio entre folds = 0.0422


+ season_ppg_home: media ponderada = 1.0229 | desvio entre folds = 0.0407


+ round_n: media ponderada = 1.0235 | desvio entre folds = 0.0419


,extra,media,folds_melhores
3,form_pts_away,1.0226,3
0,venue_form_diff,1.0226,3
7,season_ppg_home,1.0229,2
5,h2h_home_pts,1.0232,2
1,form_pts_diff,1.0233,3
8,round_n,1.0235,1
4,form_pts_home,1.0235,2
6,rest_days_home,1.0241,3
2,gd_diff,1.0241,1


In [6]:
combos = {'elo+venue': ['elo_diff', 'venue_form_diff'],
          'elo+venue+fpa': ['elo_diff', 'venue_form_diff', 'form_pts_away'],
          'elo+venue+fpa+h2h': ['elo_diff', 'venue_form_diff', 'form_pts_away', 'h2h_home_pts']}
res = {}
for name, cols in combos.items():
    for C in [0.3, 0.1, 0.03]:
        preds, _ = walk_forward(lambda: make_logreg(cols, C=C), cols)
        res[(name, C)] = summarize(preds, ys, f'{name} C={C}')

elo+venue C=0.3: media ponderada = 1.0227 | desvio entre folds = 0.0420


elo+venue C=0.1: media ponderada = 1.0226 | desvio entre folds = 0.0417


elo+venue C=0.03: media ponderada = 1.0225 | desvio entre folds = 0.0408


elo+venue+fpa C=0.3: media ponderada = 1.0224 | desvio entre folds = 0.0410


elo+venue+fpa C=0.1: media ponderada = 1.0223 | desvio entre folds = 0.0408


elo+venue+fpa C=0.03: media ponderada = 1.0222 | desvio entre folds = 0.0402


elo+venue+fpa+h2h C=0.3: media ponderada = 1.0228 | desvio entre folds = 0.0413


elo+venue+fpa+h2h C=0.1: media ponderada = 1.0227 | desvio entre folds = 0.0411


elo+venue+fpa+h2h C=0.03: media ponderada = 1.0226 | desvio entre folds = 0.0404


`venue_form_diff` e `form_pts_away` são as adições que melhoram a média; `h2h_home_pts` piora (muitos ausentes e poucos confrontos por par). O ganho é pequeno, mas aparece na combinação e com regularização maior, então promovo `elo+venue+fpa` com `C=0.03`.

In [7]:
cols = combos['elo+venue+fpa']
df, m = res[('elo+venue+fpa', 0.03)]
final = make_logreg(cols, C=0.03).fit(train[cols], train['Res'])
sub = save_submission(predict_ordered(final, test[cols]), SUB_DIR / 'submission_v04.csv')
log_experiment('v04', '+venue_form_diff +form_pts_away', ','.join(cols), 'LogisticRegression', 'C=0.03',
               df, m, 'promovido', 'submissions/submission_v04.csv')

salvo em ..\submissions\submission_v04.csv (1725, 4)


,versao,data,mudanca,features,modelo,parametros,folds,logloss_por_fold,logloss_medio,desvio,decisao,csv
0,v01,2026-09-28,baseline logreg,17 numericas,LogisticRegression,C=1,"2016,2017,2018,2019,2020,2021",2016:1.0119;2017:1.1087;2018:0.9857;2019:0.997...,1.0337,0.0455,campeao inicial,submissions/submission_v01.csv
1,v02,2026-09-28,ajuste de C,17 numericas,LogisticRegression,C=0.003,"2016,2017,2018,2019,2020,2021",2016:1.0017;2017:1.0924;2018:0.9839;2019:1.000...,1.0273,0.0403,promovido,submissions/submission_v02.csv
2,v03,2026-09-28,selecao: so elo_diff (arvores rejeitadas),elo_diff,LogisticRegression,C=0.1,"2016,2017,2018,2019,2020,2021",2016:0.9969;2017:1.0875;2018:0.9800;2019:0.989...,1.0228,0.0414,promovido,submissions/submission_v03.csv
3,v04,2026-09-28,+venue_form_diff +form_pts_away,"elo_diff,venue_form_diff,form_pts_away",LogisticRegression,C=0.03,"2016,2017,2018,2019,2020,2021",2016:0.9976;2017:1.0858;2018:0.9808;2019:0.990...,1.0222,0.0402,promovido,submissions/submission_v04.csv
